<div style="font-size: 13px; line-height: 1.2;">
  <h3>Multi-Query Retrieval – Concept</h3>

  <p><strong>What is it?</strong><br>
  Multi-query retrieval is an advanced RAG technique that <strong>generates multiple variations</strong> of the user’s question, retrieves documents for each variation, and <strong>combines the results</strong>. This helps when a single question phrasing doesn’t retrieve all relevant chunks.</p>

  <p><strong>How it works</strong></p>
  <ol>
    <li><strong>User asks a question</strong><br>
        Example: <em>“What are the common diseases in Nigeria?”</em></li>
    <li><strong>LLM generates multiple query variations</strong>
      <ul>
        <li>“What are the major public health problems in Nigeria?”</li>
        <li>“List communicable and infectious diseases prevalent in Nigeria.”</li>
        <li>“What diseases are common in Nigeria’s health sector?”</li>
      </ul>
    </li>
    <li><strong>Each variation is sent to the retriever</strong><br>
        The retriever finds relevant chunks for each version.</li>
    <li><strong>Results are combined</strong><br>
        Duplicate chunks are removed, and the union of retrieved documents is used as context.</li>
    <li><strong>LLM answers using the combined context</strong><br>
        Because more chunks are retrieved, the answer is more complete.</li>
  </ol>

  <p><strong>Why use it?</strong></p>
  <ul>
    <li><strong>Improves recall</strong> – finds relevant chunks that a single query might miss.</li>
    <li><strong>Handles ambiguity</strong> – broad or vague questions become multiple focused queries.</li>
    <li><strong>Better context</strong> – the LLM sees a richer set of sources, leading to a more accurate answer.</li>
    <li><strong>No manual metadata filtering needed</strong> – the retriever itself is queried from different angles.</li>
  </ul>

  <p><strong>Visualisation</strong></p>
  <pre style="background-color: #135316; padding: 10px; border-radius: 5px;">
User Query
   │
   ▼
[LLM] generates 3-5 query variations
   │
   ├── Variation 1 ──&gt; Retriever ──&gt; Chunks A
   ├── Variation 2 ──&gt; Retriever ──&gt; Chunks B
   └── Variation 3 ──&gt; Retriever ──&gt; Chunks C
   │
   ▼
Combine chunks (remove duplicates)
   │
   ▼
LLM generates answer using combined context
  </pre>

  <p><strong>When to use it</strong></p>
  <ul>
    <li>For <strong>broad or complex questions</strong>.</li>
    <li>When you suspect the exact wording differs between question and document.</li>
    <li>In production RAG systems where high recall is important.</li>
  </ul>

  <p><strong>Next Step</strong><br>
  We will implement multi-query retrieval in a notebook using LangChain, then integrate it into the RAG service.</p>
</div>

In [1]:
# Import necessary libraries
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_community.document_loaders import PyPDFLoader, BSHTMLLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv

load_dotenv()

print('Imports ready.')

Imports ready.


Load Relevant Documents

In [2]:
# Load only the documents we want (same as clean setup)
health_pdf = PyPDFLoader('../../04_data_ingestion_document_processing/data/nigeria_health_diseases_and_prevention.pdf').load()
crop_pdf = PyPDFLoader('../../04_data_ingestion_document_processing/data/crop_disease.pdf').load()
agri_html = BSHTMLLoader('../../04_data_ingestion_document_processing/data/agriculture.html', open_encoding='utf-8', bs_kwargs={'features': 'html.parser'}).load()
agri_txt = TextLoader('../../04_data_ingestion_document_processing/data/agriculture.txt', encoding='utf-8').load()

all_docs = health_pdf + crop_pdf + agri_html + agri_txt
print(f'Loaded {len(all_docs)} documents')

Loaded 37 documents


Split and Add Metadata

In [3]:
# Split documents into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(all_docs)

# Add metadata
for i, chunk in enumerate(chunks):
    source = chunk.metadata.get('source', '')
    file_name = source.split('\\')[-1]   # for Windows path
    chunk.metadata['file_name'] = file_name
    chunk.metadata['doc_type'] = 'pdf' if file_name.endswith('.pdf') else 'html' if file_name.endswith('.html') else 'txt'
    chunk.metadata['language'] = 'English'
    chunk.metadata['chunk_id'] = f'{file_name}_{i+1:03d}'

print(f'Created {len(chunks)} chunks')

Created 185 chunks


Build Vector Store and Base Retriever

In [4]:
# Build in-memory vector store
embeddings = OpenAIEmbeddings()
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=None
)

# Create base retriever (k=4)
base_retriever = vectorstore.as_retriever(search_kwargs={'k': 4})

print('Vector store and retriever ready.')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector store and retriever ready.


In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Create an LLM for query generation
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# 2. Prompt to generate 3 variations
multi_query_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Generate 3 different rephrased versions of the user's question to retrieve relevant documents from a vector database. Put each version on a new line."),
    ("human", "{question}")
])

# 3. Build the chain to generate query variations
generate_queries_chain = multi_query_prompt | llm | StrOutputParser()

# 4. Original query
query = 'What are Nigeria communicable and infectious diseases?'

# 5. Generate variations
generated_queries_text = generate_queries_chain.invoke({"question": query})
generated_queries = [q.strip() for q in generated_queries_text.split('\n') if q.strip()]

print('Generated queries:')
for i, q in enumerate(generated_queries, start=1):
    print(f'{i}. {q}')

# 6. Retrieve documents for each variation and combine unique chunks
all_docs = []
seen = set()
for q in generated_queries:
    docs = base_retriever.invoke(q)
    for doc in docs:
        text = doc.page_content.strip()
        if text not in seen:
            seen.add(text)
            all_docs.append(doc)

print(f'\nRetrieved {len(all_docs)} unique chunks from all queries.')

Generated queries:
1. What are the communicable and infectious diseases prevalent in Nigeria?
2. Can you provide a list of infectious and communicable diseases found in Nigeria?
3. What types of infectious diseases are common in Nigeria?

Retrieved 7 unique chunks from all queries.


Create LLM and Multi-Query Retriever

In [5]:
# LLM to generate multiple query variations
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Create a MultiQueryRetriever
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm,
)
print('MultiQueryRetriever ready')

MultiQueryRetriever ready


In [6]:
# query = 'What are Nigeria communicable and infectious diseases?'

# # Generate the multiple query variations
# generated_queries = multi_query_retriever.generate_queries(query)

# print('Generated queries:')
# for i, q in enumerate(generated_queries, start=1):
#     print(f'{i}. {q}')

Test with a Broad Question

In [7]:
query = 'What are Nigeria communicable and infectious diseases?'

docs = multi_query_retriever.invoke(query)

print(f'Retrieved {len(docs)} unique chunks using multi-query:')

for i, doc in enumerate(docs, start=1):
    print(f'{i}. {doc.page_content[:200]}')
    print(f'   Source: {doc.metadata.get("source", "unknown")}')

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Retrieved 7 unique chunks using multi-query:
1. scientific database sources, web search engines, direct observation and relevant documents from the Nigerian Ministry 
of Health. The major public health challenges Nigeria faces are infectious diseas
   Source: ../../04_data_ingestion_document_processing/data/nigeria_health_diseases_and_prevention.pdf
2. like Hypertension, Cancer, Obesity etc. Most of the 
Nigerians (young and old) die of different             
preventable diseases such as HIV/AIDS, tuberculo-
sis, malaria, vaccine preventable disease
   Source: ../../04_data_ingestion_document_processing/data/nigeria_health_diseases_and_prevention.pdf
3. Introduction Practice Points 
 Nigeria is often referred to as the "Giant of  
Africa", owing to its large population and  
economy, with approximately 182 million   
inhabitants.  
 Communicable an
   Source: ../../04_data_ingestion_document_processing/data/nigeria_health_diseases_and_prevention.pdf
4. major health problem in Nigeria

Generate an Answer with Multi-Query

In [8]:
from langchain_core.runnables import RunnablePassthrough

# Define prompt 
rag_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant. Answer the question using only the provided context. If you don\'t know, say you don\'t know.'),
    ('human', 'Context:\n{context}\n\nQuestion: {question}')
])

# Helper to format retrieved docs
def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

# Build a chain that uses multi-query retriever
multi_query_chain = (
    {
        'context': multi_query_retriever | format_docs,
        'question': RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

# Ask the question and get answer
query = 'What are Nigeria communicable and infectious diseases?'
answer = multi_query_chain.invoke(query)

print('Answer')
print(answer)

Answer
The communicable and infectious diseases in Nigeria include malaria, HIV/AIDS, tuberculosis, and vaccine-preventable diseases of childhood.
